# 03 — Sentiment Model Training
Baseline **TF-IDF + Logistic Regression** sentiment classifier trained on `data/reviews.csv` — which contains the real **NLTK `movie_reviews` corpus** (Pang & Lee, 2000 real human-written movie reviews, binary Positive/Negative), a well-known public sentiment benchmark dataset. No synthetic text is used anywhere in this notebook.

**Want retail-domain reviews instead?** Download the real Kaggle *Women's E-Commerce Clothing Reviews* dataset, keep the same two columns (`review_text`, `sentiment`), and overwrite `data/reviews.csv` — everything below runs unchanged, and you'll additionally get a real Neutral class if you bucket its 1-5 star ratings into negative/neutral/positive.

**Stretch goal**: replace this baseline with a fine-tuned DistilBERT (HuggingFace) for higher accuracy — `nlp_service.py` is model-agnostic, so only this training step and the saved artifact format would change.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pickle

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from app import config
from app.services.nlp_service import clean_text


## Load & clean the real review data

In [2]:
df = pd.read_csv(config.REVIEWS_CSV_PATH)
print(df["sentiment"].value_counts())

df["cleaned"] = df["review_text"].apply(clean_text)
df.head()


sentiment
negative    1000
positive    1000
Name: count, dtype: int64


,review_text,sentiment,cleaned
0,"plot : two teen couples go to a church party ,...",negative,plot two teen couple go church party drink dri...
1,the happy bastard's quick movie review damn t...,negative,happy bastard quick movie review damn bug got ...
2,it is movies like these that make a jaded movi...,negative,movie like make jaded movie viewer thankful in...
3,""" quest for camelot "" is warner bros . ' first...",negative,quest camelot warner bros first featurelength ...
4,synopsis : a mentally unstable man undergoing ...,negative,synopsis mentally unstable man undergoing psyc...


## Vectorize + train/test split

In [3]:
vectorizer = TfidfVectorizer(max_features=20000)
X = vectorizer.fit_transform(df["cleaned"])
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


## Train Logistic Regression classifier

In [4]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

    negative       0.85      0.80      0.82       250
    positive       0.81      0.86      0.83       250

    accuracy                           0.83       500
   macro avg       0.83      0.83      0.83       500
weighted avg       0.83      0.83      0.83       500



## Save artifacts for the API

In [5]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(config.SENTIMENT_MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

with open(config.VECTORIZER_PATH, "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved:", config.SENTIMENT_MODEL_PATH)
print("Saved:", config.VECTORIZER_PATH)


Saved: /mnt/c/Users/ndben/Downloads/smart-retail-ai/app/models/sentiment_model.pkl
Saved: /mnt/c/Users/ndben/Downloads/smart-retail-ai/app/models/vectorizer.pkl
